In [ ]:
import sys
from pathlib import Path

WOOSH_ROOT = Path("/workspace/Woosh")
sys.path.append(str(WOOSH_ROOT))

import woosh

print("Woosh import successful")
print(woosh)

In [ ]:
import torch
import torchaudio

from woosh.inference.flowmatching_sampler import flowmatching_integrate
from woosh.model.ldm import LatentDiffusionModel
from woosh.components.base import LoadConfig

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

COMPONENT_PATH = "checkpoints/Woosh-Flow"

ldm = LatentDiffusionModel(
    LoadConfig(path=COMPONENT_PATH)
)

ldm = ldm.eval().to(device)

In [ ]:
def generate_sfx(
    description,
    output_path,
    cfg=4.5,
    atol=0.001,
    rtol=0.001,
):
    batch_size = 1

    noise = torch.randn(
        batch_size,
        128,
        501,
        device=device
    )

    cond = ldm.get_cond(
        {
            "audio": None,
            "description": [description]
        },
        no_dropout=True,
        device=device,
    )

    with torch.inference_mode():
        x_fake, steps = flowmatching_integrate(
            ldm,
            noise=noise,
            cond=cond,
            cfg=cfg,
            atol=atol,
            rtol=rtol,
            return_steps=True,
            device=device,
            dtype=torch.float64,
        )

        audio = ldm.autoencoder.inverse(x_fake)

    audio = audio.cpu()

    max_abs = torch.max(torch.abs(audio[0]))
    normalization_factor = max_abs if max_abs > 1.0 else 1.0
    audio = audio[0] / normalization_factor

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    torchaudio.save(
        str(output_path),
        audio,
        sample_rate=48000,
    )

    print(f"Generated in {steps + 1} integration steps")
    print(f"Saved to {output_path}")

In [ ]:
generate_sfx(
    description="heavy wooden door creaking open",
    output_path="/workspace/storyfx/outputs/sfx_assets/test_door.wav"
)